# Create a reproducible BioASQ retrieval sample

This notebook creates and persistently stores a small retrieval sample from `BeIR/bioasq-generated-queries`. Every selected query retains its source PubMed document as its positive document. Randomly sampled, unique documents complete the retrieval corpus.

The dataset contains synthetic queries generated from their source documents. The resulting sample is useful for pipeline development and controlled model comparisons, but it is not an evaluation set of independent, expert-authored BioASQ questions.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/lohex/retrieval-benchlab.git",
                str(REPO_ROOT),
            ],
            check=True,
        )
    os.chdir(REPO_ROOT)
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets tqdm


In [ ]:
import logging
import random

import numpy as np

from src.io import (
    inspect_and_load_dataset,
    mount_google_drive,
    sample_directory,
    save_sample,
)
from src.sampling import (
    add_random_negative_documents,
    sample_queries_and_positives,
    validate_sample,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("bioasq-sample")


## Configuration

In [ ]:
DATASET_NAME = "BeIR/bioasq-generated-queries"
DATASET_CONFIG = "default"
DATASET_SPLIT = "train"

N_QUERIES = 100
N_CORPUS_DOCS = 30_000
SEED = 42

OUTPUT_ROOT = "/content/drive/MyDrive/Retreaval/data"

## Helper functions

## Final creation function

`create_bioasq_sample` orchestrates all preceding helpers. Every call loads and validates the source dataset, creates a new sample, validates it, and replaces the previously saved sample under `data/current`.

In [ ]:
def create_bioasq_sample(
    n_queries=N_QUERIES,
    n_corpus_docs=N_CORPUS_DOCS,
    seed=SEED,
    output_root=OUTPUT_ROOT,
):
    """Always create, validate, persist, and return a new BioASQ retrieval sample."""
    if n_queries <= 0 or n_corpus_docs <= 0:
        raise ValueError("n_queries and n_corpus_docs must be positive")
    if n_corpus_docs < n_queries:
        logger.warning("n_corpus_docs is smaller than n_queries; it will be raised to n_queries")

    mount_google_drive()
    output_dir = sample_directory(output_root)

    random.seed(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    logger.info("Loading source dataset")
    dataset = inspect_and_load_dataset(DATASET_NAME, DATASET_CONFIG, DATASET_SPLIT)

    logger.info("Sampling queries and positive documents")
    queries, relevant_docs, positive_corpus = sample_queries_and_positives(
        dataset, n_queries, rng
    )

    logger.info("Sampling random negative documents")
    corpus = add_random_negative_documents(dataset, positive_corpus, n_corpus_docs, rng)

    validate_sample(queries, relevant_docs, corpus, n_queries, n_corpus_docs)
    metadata = {
        "dataset_name": DATASET_NAME,
        "dataset_config": DATASET_CONFIG,
        "dataset_split": DATASET_SPLIT,
        "dataset_fingerprint": dataset._fingerprint,
        "n_queries": n_queries,
        "n_corpus_docs": len(corpus),
        "seed": seed,
        "sampling": "unique positive documents plus uniformly sampled source rows with unique document IDs",
    }
    save_sample(output_dir, queries, relevant_docs, corpus, metadata)
    return queries, relevant_docs, corpus, metadata, output_dir

## Create and save a new sample

In [ ]:
queries, relevant_docs, corpus, metadata, output_dir = create_bioasq_sample()

print(f"Saved sample: {output_dir}")
print(f"Queries: {len(queries):,}")
print(f"Corpus documents: {len(corpus):,}")
print(f"Positive relations: {sum(len(ids) for ids in relevant_docs.values()):,}")

## Browse sample examples

Change `EXAMPLE_PAGE` and rerun the following cell to browse the newly created queries and their positive documents.

In [ ]:
from IPython.display import Markdown, display

EXAMPLE_PAGE = 0
EXAMPLES_PER_PAGE = 3

query_items = list(queries.items())
n_pages = max(1, (len(query_items) + EXAMPLES_PER_PAGE - 1) // EXAMPLES_PER_PAGE)
page = EXAMPLE_PAGE % n_pages
start = page * EXAMPLES_PER_PAGE

display(Markdown(f"### Page {page + 1} of {n_pages}"))
for qid, query in query_items[start:start + EXAMPLES_PER_PAGE]:
    snippets = []
    for doc_id in sorted(relevant_docs[qid]):
        text = corpus[doc_id].replace("\n", " ")
        suffix = "…" if len(text) > 700 else ""
        snippets.append(f"**Document `{doc_id}`:** {text[:700]}{suffix}")
    display(Markdown(f"**Query `{qid}`:** {query}\n\n" + "\n\n".join(snippets)))